# CRISP-DM: Data Preparation

Na **Business Understanding** en **Data Understanding** maken we de data klaar voor analyse en modellering. In CRISP-DM heet deze fase **Data Preparation**. In dit voorbeeld gebruiken we de bankmarketingdataset uit `data/bank.csv`.

Je oefent met missende waarden, imputatie, feature engineering en one-hot encoding. We sluiten af met een preprocessing-pipeline die voorkomt dat informatie uit de testset in de training lekt. Voer de cellen van boven naar beneden uit.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/bank.csv")

print(f"Aantal rijen en kolommen: {df.shape}")
display(df.head())
df.dtypes

## 1. Missende waarden herkennen

Een echte missende waarde staat vaak als `NaN` in een dataset. In deze bankdataset komt daarnaast de tekst `unknown` voor. Dat is geen `NaN`, maar kan voor sommige variabelen wel betekenen dat de waarde onbekend is. Of je dit als ontbrekend behandelt, hangt af van de betekenis van de kolom en de onderzoeksvraag.

We laten de originele `df` intact en maken een werkkopie. Voor een aantal categorische velden behandelen we `unknown` in deze demonstratie als ontbrekend.

In [ ]:
categorical_columns = ["job", "education", "housing", "loan"]

print("Echte NaN-waarden vóór bewerking:")
display(df.isna().sum().sort_values(ascending=False).head())

print("Aantal tekstwaarden 'unknown' vóór bewerking:")
display((df[categorical_columns] == "unknown").sum())

bank_clean = df.copy()
bank_clean[categorical_columns] = bank_clean[categorical_columns].replace("unknown", np.nan)

print("Missende waarden na het herkennen van 'unknown':")
display(bank_clean[categorical_columns].isna().sum())

### Wat kun je met ontbrekende waarden doen?

- **Rijen verwijderen** (`dropna`) kan geschikt zijn als er maar weinig rijen ontbreken en het verwijderen geen vertekening veroorzaakt. Bij veel missende rijen verlies je veel data.
- **Een kolom verwijderen** kan passend zijn als een kolom grotendeels ontbreekt en niet essentieel is. Controleer eerst de inhoudelijke waarde van die kolom.
- **Imputeren** betekent een ontbrekende waarde vervangen. Voor numerieke waarden is de mediaan vaak robuuster voor uitschieters dan het gemiddelde. Voor categorische waarden kun je de meest voorkomende categorie gebruiken of expliciet `Onbekend` invullen.

De volgende kleine tabel is bewust gemaakt met missende waarden, zodat we de methoden direct kunnen vergelijken.

In [ ]:
example = pd.DataFrame({
    "leeftijd": [24, np.nan, 31, 45, np.nan],
    "opleiding": ["mbo", "hbo", np.nan, "mbo", "wo"]
})
display(example)

# Optie 1: rijen met minstens één ontbrekende waarde verwijderen
display(example.dropna())

# Optie 2: numeriek imputeren met de mediaan en categorisch met de modus
example_imputed = example.copy()
example_imputed["leeftijd"] = example_imputed["leeftijd"].fillna(example_imputed["leeftijd"].median())
example_imputed["opleiding"] = example_imputed["opleiding"].fillna(example_imputed["opleiding"].mode()[0])
display(example_imputed)

# Alternatief: onbekende categorieën expliciet herkenbaar laten
example_unknown = example.copy()
example_unknown["opleiding"] = example_unknown["opleiding"].fillna("Onbekend")
display(example_unknown)

## 2. Feature engineering

**Feature engineering** is het maken of aanpassen van variabelen op basis van bestaande data en domeinkennis. We maken hier het totaal aantal contactpogingen (`campaign` + `previous`), een indicator voor een eerste contact en leeftijdsgroepen.

Dit zijn voorbeelden, geen universeel juiste keuzes: controleer altijd of de nieuwe variabele inhoudelijk logisch is en of de benodigde informatie beschikbaar is op het moment waarop je een voorspelling wilt doen.

In [ ]:
def engineer_features(data):
    result = data.copy()
    result["total_contact_attempts"] = result["campaign"] + result["previous"]
    result["is_first_contact"] = (result["previous"] == 0).astype(int)
    result["age_group"] = pd.cut(
        result["age"],
        bins=[0, 30, 45, 60, np.inf],
        labels=["jonger dan 30", "30-44", "45-59", "60 of ouder"],
        right=False
    )
    return result

featured = engineer_features(bank_clean)
display(featured[["age", "age_group", "campaign", "previous", "total_contact_attempts", "is_first_contact"]].head())

## 3. One-hot encoding

Veel modellen kunnen niet rechtstreeks met tekstcategorieën rekenen. **One-hot encoding** maakt voor elke categorie een aparte 0/1-kolom. Zo krijgt bijvoorbeeld `marital` geen kunstmatige volgorde (zoals 1, 2, 3) die de categorieën groter of kleiner laat lijken.

Hier gebruiken we `pandas.get_dummies` op een klein voorbeeld. De volledige dataset kan veel categorieën en dus veel nieuwe kolommen opleveren.

In [ ]:
one_hot_demo = bank_clean[["job", "marital"]].head(8).fillna("Onbekend")
encoded_demo = pd.get_dummies(one_hot_demo, columns=["job", "marital"], dtype=int)

display(one_hot_demo)
display(encoded_demo)
print(f"Vorm vóór encoding: {one_hot_demo.shape}; na encoding: {encoded_demo.shape}")

## 4. Preprocessing als pipeline

In een modelleerworkflow zetten we imputatie, schalen en encoding bij voorkeur in een scikit-learn-pipeline. We splitsen eerst de data. Daarna leert `fit` de imputatiewaarden, schaalparameters en categorieën **alleen van de trainingsset**. De testset krijgt daarna alleen `transform`. Zo voorkomen we **data leakage**.

We voorspellen hier `y` (wel/geen inschrijving). `duration` laten we weg: de gespreksduur is pas na het telefoongesprek bekend en zou bij een voorspelling vóór het gesprek informatie lekken.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

model_data = engineer_features(bank_clean)
numeric_features = ["age", "campaign", "previous", "total_contact_attempts", "is_first_contact"]
categorical_features = ["age_group", "job", "marital", "education", "housing", "loan", "contact", "month"]
feature_columns = numeric_features + categorical_features

X = model_data[feature_columns]
y = model_data["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print(f"Train: {X_train.shape} -> {X_train_prepared.shape}")
print(f"Test:  {X_test.shape} -> {X_test_prepared.shape}")
print("Voorbeeldnamen van de gemaakte features:")
print(preprocessor.get_feature_names_out()[:15])

## Samenvatting en checklist

- Bekijk eerst het type, de betekenis en het aantal ontbrekende waarden per kolom.
- Behandel `unknown` alleen als ontbrekend als dat voor die variabele inhoudelijk klopt.
- Kies bewust tussen rijen/kolommen verwijderen en imputeren; leg die keuze vast.
- Maak afgeleide features die relevant zijn en gebruik geen informatie die bij voorspellen nog niet beschikbaar is.
- Gebruik one-hot encoding voor nominale categorieën zonder natuurlijke rangorde.
- Fit imputers, scalers en encoders op de trainingsdata; transformeer de testdata daarna.

### Reflectievragen
1. Wat kan er misgaan als je alle rijen met een ontbrekende waarde verwijdert?
2. Wanneer is `unknown` een betekenisvolle categorie in plaats van een ontbrekende waarde?
3. Waarom is `duration` ongeschikt als je vóór het telefoongesprek een voorspelling wilt doen?
4. Welke feature zou jij nog maken voor deze dataset, en welke domeinkennis is daarvoor nodig?